# 12 · TCN (convolucional)

Rede convolucional temporal (Bai, Kolter & Koltun, 2018).

Desenho adotado:
- blocos residuais com duas convoluções **causais e dilatadas** (dilatação 1, 2, 4, 8, …), `kernel=3`, 64 canais, *weight norm*, ReLU e dropout 0.1;
- o número de blocos é o **menor que faz o campo receptivo cobrir a janela inteira**. Com kernel k e n blocos, o campo receptivo é 1 + 2(k − 1)(2ⁿ − 1). Isso dá 5 blocos para L = 96 e 8 para L = 720 (a célula abaixo mostra). Assim, se a TCN ignorar o passado distante, é por aprendizado, não por construção;
- saída: o último passo da sequência passa por `Linear(64, H)`.

In [1]:
# Configuração: igual em todos os notebooks
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJETO = '/content/drive/MyDrive/RN-Projeto'
except ImportError:          # fora do Colab (ex.: rodando no computador)
    PROJETO = os.path.abspath('.')
sys.path.insert(0, os.path.join(PROJETO, 'src'))
print('Pasta do projeto:', PROJETO)

import importlib
import rn_utils as ru
importlib.reload(ru)        # pega a versão mais nova se o 02 foi rodado de novo
ru.configurar(PROJETO)

Mounted at /content/drive
Pasta do projeto: /content/drive/MyDrive/RN-Projeto
Projeto: /content/drive/MyDrive/RN-Projeto | dispositivo: cpu


## Modelo

`criar(L, H)` devolve um `nn.Module` que recebe `[lote, L, 1]` e devolve `[lote, H, 1]`. A normalização por janela (RevIN) é aplicada por fora, pelo `ru.ComRevIN`, igual para todos os modelos.

In [2]:
import torch
import torch.nn as nn
from torch.nn.utils.parametrizations import weight_norm

class Cortar(nn.Module):
    """Remove o preenchimento extra à direita: torna a convolução causal."""
    def __init__(self, n):
        super().__init__()
        self.n = n

    def forward(self, x):
        return x[:, :, :-self.n] if self.n else x

class BlocoTemporal(nn.Module):
    def __init__(self, c_in, c_out, k, dilatacao, dropout):
        super().__init__()
        pad = (k - 1) * dilatacao
        self.rede = nn.Sequential(
            weight_norm(nn.Conv1d(c_in, c_out, k, padding=pad, dilation=dilatacao)), Cortar(pad),
            nn.ReLU(), nn.Dropout(dropout),
            weight_norm(nn.Conv1d(c_out, c_out, k, padding=pad, dilation=dilatacao)), Cortar(pad),
            nn.ReLU(), nn.Dropout(dropout))
        self.atalho = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else nn.Identity()
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.rede(x) + self.atalho(x))

def campo_receptivo(niveis, k):
    return 1 + 2 * (k - 1) * (2 ** niveis - 1)

def niveis_necessarios(L, k):
    n = 1
    while campo_receptivo(n, k) < L:
        n += 1
    return n

class TCN(nn.Module):
    def __init__(self, L, H, canais=64, k=3, dropout=0.1):
        super().__init__()
        self.niveis = niveis_necessarios(L, k)
        self.rede = nn.Sequential(*[
            BlocoTemporal(1 if i == 0 else canais, canais, k, 2 ** i, dropout) for i in range(self.niveis)])
        self.saida = nn.Linear(canais, H)

    def forward(self, x):                      # [B, L, 1] → [B, H, 1]
        h = self.rede(x.transpose(1, 2))       # [B, canais, L]
        return self.saida(h[:, :, -1])[..., None]

def criar(L, H):
    return TCN(L, H)

for L in (96, 192, 336, 512, 720):
    n = niveis_necessarios(L, 3)
    print(f'L={L}: {n} blocos, campo receptivo {campo_receptivo(n, 3)} passos')

L=96: 5 blocos, campo receptivo 125 passos
L=192: 6 blocos, campo receptivo 253 passos
L=336: 7 blocos, campo receptivo 509 passos
L=512: 8 blocos, campo receptivo 1021 passos
L=720: 8 blocos, campo receptivo 1021 passos


In [3]:
# Verificação do formato de entrada e saída
import torch
for L in (96, 720):
    m = criar(L, ru.HORIZONTE)
    saida = m(torch.randn(4, L, 1))
    assert saida.shape == (4, ru.HORIZONTE, 1), saida.shape
    print(f'L={L}: saída {tuple(saida.shape)} ✓ | {sum(p.numel() for p in m.parameters()):,} parâmetros')

L=96: saída (4, 96, 1) ✓ | 118,432 parâmetros
L=720: saída (4, 96, 1) ✓ | 192,928 parâmetros


## Treino e avaliação

O learning rate é escolhido pela validação (L = 336, seed 0) entre {1e-3, 3e-4, 1e-4}. Depois o modelo é treinado em todas as janelas e seeds das duas bases e avaliado no teste, com a janela original e com as quatro janelas embaralhadas. As execuções já salvas em `resultados/` são apenas recarregadas e listadas.

In [4]:
ru.rodar_grade('TCN', criar)

lr escolhido para TCN em ETTh1: 0.0003  (val: {'0.001': 0.0931, '0.0003': 0.0929, '0.0001': 0.0932})
  · ETTh1_TCN_L96_H96_lr0.0003_s0 (salvo) | val 0.0994 | teste MSE 0.0559 MAE 0.1799 | embaralhar_tudo ×1.33 | 9 épocas, 14s (cuda)
  · ETTh1_TCN_L96_H96_lr0.0003_s1 (salvo) | val 0.1001 | teste MSE 0.0589 MAE 0.1885 | embaralhar_tudo ×1.56 | 16 épocas, 25s (cuda)
  · ETTh1_TCN_L96_H96_lr0.0003_s2 (salvo) | val 0.1014 | teste MSE 0.0561 MAE 0.1795 | embaralhar_tudo ×1.33 | 9 épocas, 14s (cuda)
  · ETTh1_TCN_L192_H96_lr0.0003_s0 (salvo) | val 0.0891 | teste MSE 0.0547 MAE 0.1791 | embaralhar_tudo ×1.49 | 9 épocas, 19s (cuda)
  · ETTh1_TCN_L192_H96_lr0.0003_s1 (salvo) | val 0.0889 | teste MSE 0.0544 MAE 0.1783 | embaralhar_tudo ×1.55 | 10 épocas, 21s (cuda)
  · ETTh1_TCN_L192_H96_lr0.0003_s2 (salvo) | val 0.0891 | teste MSE 0.0551 MAE 0.1803 | embaralhar_tudo ×1.60 | 10 épocas, 21s (cuda)
  · ETTh1_TCN_L336_H96_lr0.0003_s0 (salvo) | val 0.0929 | teste MSE 0.0553 MAE 0.1813 | embaralhar_tu

## Resumo (média das 3 seeds)

`×tudo`, `×antigo`, `×recente` e `×blocos` = MSE com a janela embaralhada / MSE original. ×1,00 significa que o modelo ignora aquela ordem.

In [5]:
df = ru.carregar_resultados()
r = df[df.modelo == 'TCN']
resumo = r.groupby(['dataset', 'L']).agg(**{
    'MSE val': ('mse_val', 'mean'), 'MSE teste': ('original_mse', 'mean'), 'dp': ('original_mse', 'std'),
    'MAE teste': ('original_mae', 'mean'), 'MSE sem suspeitos': ('original_mse_ss', 'mean'),
    **{'×' + p.replace('embaralhar_', ''): (f'{p}_razao', 'mean') for p in ru.PERTURBACOES},
    'épocas': ('epocas_rodadas', 'mean'), 'tempo (s)': ('tempo_treino_s', 'mean')})
resumo.round(4)

MSE val  MSE teste      dp  MAE teste  MSE sem suspeitos   ×tudo  \
dataset L                                                                       
ETTh1   96    0.1003     0.0570  0.0017     0.1826             0.0560  1.4056   
        192   0.0890     0.0547  0.0004     0.1792             0.0561  1.5452   
        336   0.0939     0.0552  0.0006     0.1803             0.0576  1.5200   
        512   0.0968     0.0544  0.0010     0.1794             0.0566  1.4063   
        720   0.0966     0.0573  0.0004     0.1844             0.0598  1.4120   
Weather 96    0.5398     0.4890  0.0024     0.4810             0.4890  1.4336   
        336   0.4643     0.4152  0.0147     0.4465             0.4152  1.7390   
        720   0.4921     0.4032  0.0061     0.4463             0.4032  1.8766   

             ×antigo  ×recente  ×blocos   épocas  tempo (s)  
dataset L                                                    
ETTh1   96    1.0844    1.1208   1.3334  11.3333    17.3727  
        192   1.0610    1.1867   1.5192   9.6667    20.5656  
        336   1.0222    1.2351   1.5525   8.0000    25.6982  
        512   0.9564    1.2859   1.4930   6.6667    34.7356  
        720   0.9424    1.3321   1.5024   7.0000    47.5902  
Weather 96    1.1360    1.1092   1.3935  26.0000   174.8769  
        336   1.0502    1.4035   1.8134  19.0000   269.0924  
        720   0.9901    1.4706   2.0151  13.0000   379.5954